# Day 2 · Session 1B — LoRA Hands-on  
## Teaching a Model a Consistent Faculty-Note Style

This notebook demonstrates one clear idea:

> **Fine-tuning is useful when we want consistent behaviour and format—not merely additional facts.**

We will teach a pretrained model to answer in this structure:

```text
FACULTY NOTE

Definition:
...

Example:
...

Key Takeaway:
...

Reflection Question:
...
```

### Learning outcomes

By the end of this notebook, you will be able to:

1. Compare a base model before and after LoRA.
2. See how few parameters are actually trained.
3. Train and save a LoRA adapter.
4. Reload the adapter without retraining.
5. Explain the difference between changing behaviour and adding knowledge.

> **Recommended runtime:** Google Colab with a T4 GPU.  
> It can also run in VS Code/Jupyter, but CPU or Apple Silicon training will be slower.

## 1. Setup

For Google Colab, uncomment the installation line below.

For the FDP repository, install dependencies once from the project root:

```bash
python -m pip install -r requirements.txt
```

In [1]:
# For Google Colab only:
# %pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" "accelerate>=0.33" sentencepiece

In [1]:
import os
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

Device: mps
PyTorch: 2.13.0


## 2. Load the base model

We use **FLAN-T5 Base** because it gives a more convincing before/after demonstration than FLAN-T5 Small.

We are not training the whole model. LoRA will add only a small set of trainable adapter parameters.

In [2]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
base_model.to(DEVICE)
base_model.eval()

print("Loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded: google/flan-t5-base


## 3. Deterministic generation helper

We keep generation settings identical before and after training.

The repetition controls are important for smaller models.

In [3]:
def generate_answer(model, question, max_new_tokens=140):
    prompt = (
        "Answer the following question clearly for engineering students.\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

## 4. Test the base model before LoRA

These questions are not included in the training dataset.

In [4]:
TEST_QUESTIONS = [
    "Explain bias and variance in machine learning.",
    "Explain the difference between training accuracy and validation accuracy.",
    "Explain why model evaluation is necessary.",
]

before_outputs = {}

for question in TEST_QUESTIONS:
    answer = generate_answer(base_model, question)
    before_outputs[question] = answer

    print("QUESTION:", question)
    print("\nBEFORE LoRA:\n", answer)
    print("\n" + "=" * 100 + "\n")

QUESTION: Explain bias and variance in machine learning.

BEFORE LoRA:
 Machine learning is the study of how a machine learns information.


QUESTION: Explain the difference between training accuracy and validation accuracy.

BEFORE LoRA:
 Training accuracy is a measure of the accuracy of a product or process. Validation accuracy is the ability of the product to be validated by a third party.


QUESTION: Explain why model evaluation is necessary.

BEFORE LoRA:
 Models can be used to measure the quality of a product.




## 5. Create a consistent behaviour dataset

Every target response follows the same teaching structure.

The dataset is designed to teach **format and behaviour**, not a large body of new knowledge.

In [5]:
training_examples = [{'question': 'Explain overfitting in machine learning.', 'answer': 'FACULTY NOTE\n\nDefinition:\nOverfitting occurs when a model learns the training data too closely, including noise, and performs poorly on unseen data.\n\nExample:\nA student memorises previous exam answers but struggles when the questions are reworded.\n\nKey Takeaway:\nGood training performance alone does not guarantee good generalisation.\n\nReflection Question:\nHow would you detect overfitting using training and validation curves?'}, {'question': 'Explain underfitting in machine learning.', 'answer': "FACULTY NOTE\n\nDefinition:\nUnderfitting occurs when a model is too simple to learn the important patterns in the data.\n\nExample:\nUsing a straight line to model a strongly curved relationship may produce poor results on both training and test data.\n\nKey Takeaway:\nUnderfitting usually causes high bias and weak performance everywhere.\n\nReflection Question:\nWhat changes could increase the model's learning capacity?"}, {'question': 'Explain precision in classification.', 'answer': 'FACULTY NOTE\n\nDefinition:\nPrecision measures how many predicted positive cases were actually positive.\n\nExample:\nIf a spam filter marks 10 emails as spam and 8 are truly spam, precision is 8 out of 10.\n\nKey Takeaway:\nPrecision matters when false positives are costly.\n\nReflection Question:\nIn which application would a false positive create serious consequences?'}, {'question': 'Explain recall in classification.', 'answer': 'FACULTY NOTE\n\nDefinition:\nRecall measures how many actual positive cases were correctly identified.\n\nExample:\nIf 20 patients have a disease and the model detects 16, recall is 16 out of 20.\n\nKey Takeaway:\nRecall matters when missing a positive case is costly.\n\nReflection Question:\nWhy is recall often important in medical screening?'}, {'question': 'Explain the F1-score.', 'answer': 'FACULTY NOTE\n\nDefinition:\nThe F1-score is the harmonic mean of precision and recall.\n\nExample:\nA classifier with balanced precision and recall will usually have a stronger F1-score than one that performs well on only one measure.\n\nKey Takeaway:\nUse F1 when both false positives and false negatives matter.\n\nReflection Question:\nWhy might accuracy be misleading for an imbalanced dataset?'}, {'question': 'Explain cross-validation.', 'answer': 'FACULTY NOTE\n\nDefinition:\nCross-validation evaluates a model across multiple train-validation splits.\n\nExample:\nIn five-fold cross-validation, the dataset is divided into five parts and each part becomes the validation set once.\n\nKey Takeaway:\nCross-validation gives a more reliable estimate than a single split.\n\nReflection Question:\nWhen might cross-validation be too expensive?'}, {'question': 'Explain regularisation.', 'answer': 'FACULTY NOTE\n\nDefinition:\nRegularisation discourages a model from becoming unnecessarily complex.\n\nExample:\nL2 regularisation penalises very large model weights.\n\nKey Takeaway:\nRegularisation helps improve generalisation and reduce overfitting.\n\nReflection Question:\nWhat may happen if regularisation is made too strong?'}, {'question': 'Explain gradient descent.', 'answer': 'FACULTY NOTE\n\nDefinition:\nGradient descent updates model parameters in the direction that reduces loss.\n\nExample:\nIt is similar to walking downhill by repeatedly choosing the steepest downward direction.\n\nKey Takeaway:\nLearning rate controls the size of each update.\n\nReflection Question:\nWhat happens when the learning rate is too high?'}, {'question': 'Explain a confusion matrix.', 'answer': 'FACULTY NOTE\n\nDefinition:\nA confusion matrix summarises correct and incorrect classification outcomes.\n\nExample:\nIt records true positives, true negatives, false positives, and false negatives.\n\nKey Takeaway:\nThe matrix reveals error types hidden by a single accuracy score.\n\nReflection Question:\nWhich error type is most serious for your use case?'}, {'question': 'Explain data leakage.', 'answer': 'FACULTY NOTE\n\nDefinition:\nData leakage occurs when information unavailable at prediction time enters model training.\n\nExample:\nUsing a final exam score to predict whether a student will pass the same exam creates leakage.\n\nKey Takeaway:\nLeakage produces unrealistically strong evaluation results.\n\nReflection Question:\nHow can preprocessing accidentally leak validation information?'}, {'question': 'Explain class imbalance.', 'answer': 'FACULTY NOTE\n\nDefinition:\nClass imbalance occurs when some target classes have far more examples than others.\n\nExample:\nA fraud dataset may contain thousands of normal transactions but very few fraudulent ones.\n\nKey Takeaway:\nHigh accuracy can hide poor minority-class detection.\n\nReflection Question:\nWhich evaluation metrics are useful for imbalanced data?'}, {'question': 'Explain feature scaling.', 'answer': 'FACULTY NOTE\n\nDefinition:\nFeature scaling places numerical features on comparable ranges.\n\nExample:\nAge may range from 18 to 80 while annual income may range into lakhs.\n\nKey Takeaway:\nScaling is especially important for distance-based and gradient-based algorithms.\n\nReflection Question:\nWhich algorithms are less sensitive to feature scaling?'}]

prefixes = [
    "Explain clearly:",
    "Teach an engineering student:",
    "Give a classroom explanation:",
]

expanded_examples = []
for item in training_examples:
    expanded_examples.append(item)

    for prefix in prefixes:
        expanded_examples.append({
            "question": f"{prefix} {item['question']}",
            "answer": item["answer"],
        })

random.shuffle(expanded_examples)

dataset = Dataset.from_list(expanded_examples)
split = dataset.train_test_split(test_size=0.15, seed=SEED)

train_dataset = split["train"]
eval_dataset = split["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))

Training examples: 40
Validation examples: 8


## 6. Tokenise the dataset

The question becomes the model input.  
The Faculty Note becomes the expected output.

In [6]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 220

def preprocess(batch):
    prompts = [
        (
            "Answer the following question clearly for engineering students.\n"
            f"Question: {question}\n"
            "Answer:"
        )
        for question in batch["question"]
    ]

    model_inputs = tokenizer(
        prompts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["answer"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_eval = eval_dataset.map(
    preprocess,
    batched=True,
    remove_columns=eval_dataset.column_names,
)

print(tokenized_train)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 40
})


## 7. Add LoRA adapters

The base model weights remain frozen. Only the adapter parameters are trained.

In [7]:
model_for_lora = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"],
    bias="none",
)

lora_model = get_peft_model(model_for_lora, lora_config)
lora_model.to(DEVICE)

lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


### Wow moment

The output above shows that only a small fraction of parameters are trainable.

> **LoRA learns a compact update instead of rewriting the entire model.**

## 8. Train the adapter

The settings below are deliberately conservative to reduce overfitting and repetition.

In [8]:
OUTPUT_DIR = "./faculty_note_lora_output"
ADAPTER_DIR = "./faculty_note_lora_adapter"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.10,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model,
    label_pad_token_id=-100,
)

trainer = Seq2SeqTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    processing_class=tokenizer,
)

train_result = trainer.train()
train_result

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,8.694529,3.931528
2,8.699404,3.902338
3,8.599278,3.879097
4,8.561545,3.869177


/Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=20, training_loss=8.638689041137695, metrics={'train_runtime': 13.8449, 'train_samples_per_second': 11.557, 'train_steps_per_second': 1.445, 'total_flos': 5671670906880.0, 'train_loss': 8.638689041137695, 'epoch': 4.0})

## 9. Compare before and after LoRA

We use the same held-out questions.

In [9]:
lora_model.eval()
after_outputs = {}

for question in TEST_QUESTIONS:
    answer = generate_answer(lora_model, question)
    after_outputs[question] = answer

    print("QUESTION:", question)
    print("\nBEFORE LoRA:\n", before_outputs[question])
    print("\nAFTER LoRA:\n", answer)
    print("\n" + "=" * 100 + "\n")

QUESTION: Explain bias and variance in machine learning.

BEFORE LoRA:
 Machine learning is the study of how a machine learns information.

AFTER LoRA:
 Machine learning is the study of how a machine learns information.


QUESTION: Explain the difference between training accuracy and validation accuracy.

BEFORE LoRA:
 Training accuracy is a measure of the accuracy of a product or process. Validation accuracy is the ability of the product to be validated by a third party.

AFTER LoRA:
 Training accuracy is a measure of the accuracy of a product or process. Validation accuracy is the ability of the product to be validated by a third party.


QUESTION: Explain why model evaluation is necessary.

BEFORE LoRA:
 Models can be used to measure the quality of a product.

AFTER LoRA:
 Model evaluation is a process that evaluates the properties of a model.




## 10. Simple behaviour evaluation

We check whether the target teaching sections appear.

Production evaluation should also assess factual correctness and robustness.

In [11]:
REQUIRED_SECTIONS = [
    "FACULTY NOTE",
    "Definition:",
    "Example:",
    "Key Takeaway:",
    "Reflection Question:",
]

def format_score(text):
    matched = sum(section in text for section in REQUIRED_SECTIONS)
    return matched, len(REQUIRED_SECTIONS)

for question, output in after_outputs.items():
    matched, total = format_score(output)
    print(f"{matched}/{total} sections matched — {question}")

0/5 sections matched — Explain bias and variance in machine learning.
0/5 sections matched — Explain the difference between training accuracy and validation accuracy.
0/5 sections matched — Explain why model evaluation is necessary.


## 11. Save the LoRA adapter

Only the learned adapter and tokenizer are saved.

In [ ]:
lora_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Adapter saved to:", ADAPTER_DIR)

total_size = 0
for root, _, files in os.walk(ADAPTER_DIR):
    for filename in files:
        file_path = os.path.join(root, filename)
        total_size += os.path.getsize(file_path)

print(f"Saved adapter folder size: {total_size / (1024**2):.2f} MB")

## 12. Reload without retraining

This completes the lifecycle:

> **Train → Save → Reload → Use**

In [ ]:
reloaded_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
reloaded_model = PeftModel.from_pretrained(reloaded_base, ADAPTER_DIR)
reloaded_model.to(DEVICE)
reloaded_model.eval()

question = "Explain why a validation set is important."
print(generate_answer(reloaded_model, question))

## 13. What did LoRA change?

LoRA changed the model's tendency to follow a consistent response format.

It did not:

- create a new foundation model
- guarantee factual correctness
- add fresh institutional knowledge
- replace evaluation

Use **RAG** for changing knowledge.  
Use fine-tuning when consistent behaviour or format is the real requirement.

## 14. Participant activity

Change the target structure to:

```text
CONCEPT CARD

Simple Definition:
Analogy:
Common Mistake:
One-Minute Activity:
```

Then retrain and evaluate the output on held-out questions.

# Key takeaways

1. Fine-tuning should solve a measurable behaviour problem.
2. Dataset consistency matters more than increasing epochs.
3. LoRA trains a small fraction of the parameters.
4. Held-out prompts are essential for fair evaluation.
5. Saving the adapter avoids retraining every time.
6. Evaluate both format and factual quality.